In [ ]:
import os
import cv2
import numpy as np
import pywt
from tqdm import tqdm
from sklearn.model_selection import train_test_split

RAW_DATASET_ROOT = "E:/2025-2026/Project/code1/archive/aptos_sorted"
OUTPUT_DIR = "E:/2025-2026/Project/code1/APTOS_processed"
IMG_SIZE = 224 

CLASSES_MAP = {
    0: "No_DR",
    1: "DR/mild1",
    2: "DR/moderate2",
    3: "DR/severe4",
    4: "DR/proliferative3"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)


def improve_robustness(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    blur = cv2.GaussianBlur(img_clahe, (0, 0), sigmaX=IMG_SIZE / 30)
    img_final = cv2.addWeighted(img_clahe, 4, blur, -4, 128)
    return img_final

def get_wavelet_auxiliary_channels(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    coeffs = pywt.dwt2(gray.astype(np.float32), 'haar')
    LL, (LH, HL, HH) = coeffs
    channels = []
    for subband in [LL, LH, HL, HH]:
        resized = cv2.resize(subband, (IMG_SIZE, IMG_SIZE))
        norm = (resized - np.mean(resized)) / (np.std(resized) + 1e-6)
        norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-6)
        channels.append(norm)
    return np.stack(channels, axis=-1)

def preprocess_pipeline(path):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 10
    if mask.any():
        coords = np.column_stack(np.where(mask))
        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        img = img[int(ymin):int(ymax)+1, int(xmin):int(xmax)+1]
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_enhanced = improve_robustness(img)
    wavelet_feats = get_wavelet_auxiliary_channels(img_enhanced)
    
    img_norm = img_enhanced.astype(np.float32) / 255.0
    return np.concatenate([img_norm, wavelet_feats], axis=-1)



def get_all_filepaths():
    all_files = []
    all_labels = []
    for label_idx, folder_path in CLASSES_MAP.items():
        full_path = os.path.join(RAW_DATASET_ROOT, folder_path)
        if not os.path.exists(full_path):
            print(f"Warning: Folder {full_path} not found.")
            continue
        files = [os.path.join(full_path, f) for f in os.listdir(full_path) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        all_files.extend(files)
        all_labels.extend([label_idx] * len(files))
    return np.array(all_files), np.array(all_labels)

def process_and_save_group(file_paths, labels, set_name, is_train=False):
    """
    set_name: 'train', 'val', or 'test'
    is_train: If true, oversampling balancing is performed
    """
    final_files = []
    final_labels = []

    if is_train:
        
        unique_labels = np.unique(labels)
        counts = [np.sum(labels == l) for l in unique_labels]
        target_count = max(counts)
        print(f"Balancing training set to {target_count} samples per class...")
        
        for l in unique_labels:
            idx_of_label = np.where(labels == l)[0]
        
            resampled_idx = np.random.choice(idx_of_label, target_count, replace=True)
            for i in resampled_idx:
                final_files.append(file_paths[i])
                final_labels.append(l)
    else:
        final_files = file_paths
        final_labels = labels

    
    X_list, y_list = [], []
    print(f"Processing {set_name} set ({len(final_files)} images)...")
    
    for i in tqdm(range(len(final_files))):
        data = preprocess_pipeline(final_files[i])
        if data is not None:
            
            if is_train and i >= len(file_paths): 
                if np.random.rand() > 0.5: data = np.flip(data, axis=1)

            X_list.append(data)
            y_list.append(final_labels[i])
            
    
    X_arr = np.array(X_list, dtype=np.float32)
    y_arr = np.array(y_list, dtype=np.int32)
    
    y_onehot = np.eye(5)[y_arr]

    np.save(os.path.join(OUTPUT_DIR, f"X_{set_name}.npy"), X_arr)
    np.save(os.path.join(OUTPUT_DIR, f"y_{set_name}.npy"), y_onehot)
    print(f"Successfully saved {set_name} set. Shape: {X_arr.shape}")



if __name__ == "__main__":
    
    paths, labels = get_all_filepaths()
    
    
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        paths, labels, test_size=0.15, stratify=labels, random_state=42
    )
    
    
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_val_paths, train_val_labels, test_size=0.15, stratify=train_val_labels, random_state=42
    )
    
    
    process_and_save_group(train_paths, train_labels, "train", is_train=True)
    process_and_save_group(val_paths, val_labels, "val", is_train=False)
    process_and_save_group(test_paths, test_labels, "test", is_train=False)

    print("\n--- All Done! ---")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, cohen_kappa_score, 
                             f1_score, roc_auc_score)


DATA_DIR = "E:/2025-2026/Project/code1/APTOS_processed"
SAVE_DIR = "./results_final"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 16  
EPOCHS = 40
LEARNING_RATE = 1e-4

def load_data():
    print("Loading 7-Channel data (RGB + Wavelet)...")
    X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
    y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
    X_val = np.load(os.path.join(DATA_DIR, "X_val.npy"))
    y_val = np.load(os.path.join(DATA_DIR, "y_val.npy"))
    X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
    y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))
    return X_train, y_train, X_val, y_val, X_test, y_test



def focal_loss_with_smoothing(gamma=2., alpha=0.25, smoothing=0.1):
    
    def loss(y_true, y_pred):
        y_true = y_true * (1.0 - smoothing) + (smoothing / 5.0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        focal_weight = alpha * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(focal_weight * (-y_true * tf.math.log(y_pred)), axis=1)
    return loss

def se_block(x, ratio=8):
    
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return layers.multiply([x, layers.Reshape((1, 1, filters))(se)])

def residual_inception_module(x, filters):
    
    shortcut = layers.Conv2D(filters, 1, padding='same')(x)
    shortcut = layers.BatchNormalization()(shortcut)
    
    
    b1 = layers.SeparableConv2D(filters // 4, 1, padding='same', activation='relu')(x)
    b2 = layers.SeparableConv2D(filters // 4, 3, padding='same', activation='relu')(x)
    b3 = layers.SeparableConv2D(filters // 4, 5, padding='same', activation='relu')(x)
    b4 = layers.MaxPooling2D(3, strides=1, padding='same')(x)
    b4 = layers.Conv2D(filters // 4, 1, padding='same', activation='relu')(b4)
    
    m = layers.Concatenate()([b1, b2, b3, b4])
    m = layers.BatchNormalization()(m)
    return layers.Activation('relu')(layers.add([m, shortcut]))


def build_model():
    inputs = layers.Input(shape=(224, 224, 7))
    
    x = layers.SeparableConv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    
    for f in [64, 128, 256]:
        x = residual_inception_module(x, f)
        x = se_block(x)
        x = layers.MaxPooling2D()(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(5, activation='softmax')(x)
    
    return models.Model(inputs, outputs)



if __name__ == "__main__":
    X_train, y_train, X_val, y_val, X_test, y_test = load_data()
    
    model = build_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=focal_loss_with_smoothing(),
        metrics=['accuracy']
    )

    
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(SAVE_DIR, "best_model.h5"), 
            save_best_only=True, monitor='val_accuracy'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)
    ]

    print("\n[INFO] Beginning training...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks
    )

    # 保存训练历史 CSV
    pd.DataFrame(history.history).to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)
    print(f"\n[SUCCESS] Training history and model saved to {SAVE_DIR}")

    
    print("\n[INFO] Generating test set report...")
    
    
    best_model = models.load_model(
        os.path.join(SAVE_DIR, "best_model.h5"),
        custom_objects={'loss': focal_loss_with_smoothing()}
    )
    
    y_pred_probs = best_model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test, axis=1)

    
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    f1_m = f1_score(y_true, y_pred, average='macro')
    
    print(f">>> Quadratic Kappa Score: {kappa:.4f}")
    print(f">>> Macro F1-Score: {f1_m:.4f}")

    
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=["Normal", "Mild", "Moderate", "Severe", "Prolif"],
                yticklabels=["Normal", "Mild", "Moderate", "Severe", "Prolif"])
    plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"))
    print(f"[INFO] Evalution metrics and confusion matrix saved to ")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from tensorflow.keras.models import load_model


SAVE_DIR = "./results_final"
DATA_DIR = "E:/2025-2026/Project/code1/APTOS_processed"
target_names = ["Normal", "Mild", "Moderate", "Severe", "Prolif"]


history_df = pd.read_csv(os.path.join(SAVE_DIR, "training_history.csv"))


def plot_learning_curves(df):
    plt.figure(figsize=(16, 6))
    
    # --- Loss Curve ---
    plt.subplot(1, 2, 1)
    plt.plot(df['loss'], label='Train Loss', color='#1f77b4', lw=2)
    plt.plot(df['val_loss'], label='Val Loss', color='#ff7f0e', lw=2, linestyle='--')
    

    min_val_loss = df['val_loss'].min()
    best_epoch = df['val_loss'].idxmin()
    plt.annotate(f'Best Loss: {min_val_loss:.4f}', 
                 xy=(best_epoch, min_val_loss), 
                 xytext=(best_epoch + 2, min_val_loss + 0.05),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))
    
    plt.title('Loss Convergence Analysis', fontsize=14)
    plt.xlabel('Epochs'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3)

    
    plt.subplot(1, 2, 2)
    plt.plot(df['accuracy'], label='Train Acc', color='#2ca02c', lw=2)
    plt.plot(df['val_accuracy'], label='Val Acc', color='#d62728', lw=2, linestyle='--')
    
    
    max_val_acc = df['val_accuracy'].max()
    best_acc_epoch = df['val_accuracy'].idxmax()
    plt.annotate(f'Best Acc: {max_val_acc:.4f}', 
                 xy=(best_acc_epoch, max_val_acc), 
                 xytext=(best_acc_epoch - 10, max_val_acc - 0.1),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))
    
    plt.title('Accuracy Growth Curve', fontsize=14)
    plt.xlabel('Epochs'); plt.ylabel('Accuracy')
    plt.legend(); plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "annotated_learning_curves.png"), dpi=300)
    plt.show()


def plot_advanced_metrics():
    def focal_loss_with_smoothing(gamma=2., alpha=0.25, smoothing=0.1):
        def loss(y_true, y_pred):
            y_true = y_true * (1.0 - smoothing) + (smoothing / 5.0)
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            focal_weight = alpha * tf.pow(1.0 - y_pred, gamma)
            return tf.reduce_sum(focal_weight * (-y_true * tf.math.log(y_pred)), axis=1)
        return loss

    X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
    y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))
    
    model = load_model(os.path.join(SAVE_DIR, "best_model.h5"), 
                       custom_objects={'loss': focal_loss_with_smoothing()})
    
    y_score = model.predict(X_test)
    n_classes = 5

    
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{target_names[i]} (AUC = {roc_auc:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.title('Multi-class ROC-AUC Analysis', fontsize=14)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.2)
    plt.savefig(os.path.join(SAVE_DIR, "annotated_roc_auc.png"), dpi=300)
    plt.show()

   
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_test[:, i], y_score[:, i])
        
        avg_recall = np.mean(recall)
        plt.plot(recall, precision, lw=2, label=f'{target_names[i]} (Avg Recall = {avg_recall:.2f})')
    
    plt.title('Precision-Recall Analysis (Clinical Recall Focus)', fontsize=14)
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.legend(loc="lower left", fontsize=10)
    plt.grid(alpha=0.2)
    plt.savefig(os.path.join(SAVE_DIR, "annotated_pr_recall.png"), dpi=300)
    plt.show()

if __name__ == "__main__":
    plot_learning_curves(history_df)
    plot_advanced_metrics()

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt


X_eval_small = X_eval[:5]
X_bg = X_train[:10]

def flatten(x):
    return x.reshape(x.shape[0], -1)

X_eval_f = flatten(X_eval_small)
X_bg_f   = flatten(X_bg)


def f(x):
    x = x.reshape(-1, 224, 224, 7)
    preds = model.predict(x)
    return np.max(preds, axis=1)   


print("Building explainer...")
explainer = shap.KernelExplainer(f, X_bg_f)

print("Computing SHAP...")
shap_values = explainer.shap_values(X_eval_f, nsamples=30)

sv = np.array(shap_values)  

print("Raw SHAP shape:", sv.shape)


def reduce_to_channels(shap_vals):
    shap_vals = shap_vals.reshape(-1, 224,224,7)
    return np.mean(np.abs(shap_vals), axis=(1,2))

def reduce_input(x):
    x = x.reshape(-1,224,224,7)
    return np.mean(x, axis=(1,2))

sv_reduced = reduce_to_channels(sv)
X_reduced  = reduce_input(X_eval_f)

print("Reduced SHAP:", sv_reduced.shape)
print("Reduced X:", X_reduced.shape)

feature_names = ["R","G","B","LL","LH","HL","HH"]


shap.summary_plot(
    sv_reduced,
    X_reduced,
    feature_names=feature_names,
    plot_type="bar"
)


shap.summary_plot(
    sv_reduced,
    X_reduced,
    feature_names=feature_names
)


shap.initjs()

shap.force_plot(
    explainer.expected_value,
    sv_reduced[0],
    X_reduced[0],
    feature_names=feature_names,
    matplotlib=True
)


shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    sv_reduced[0],
    feature_names=feature_names
)

In [ ]:
import os
import cv2
import numpy as np
import pywt
from tqdm import tqdm


RAW_DATASET2_DIR = "E:/2025-2026/Project/code1/dataset" 
OUTPUT_DIR2 = "E:/2025-2026/Project/code1/Dataset2_processed"
IMG_SIZE = 224


CLASSES_MAP = {
    "normal": 0,                
    "diabetic_retinopathy": 3   
}

os.makedirs(OUTPUT_DIR2, exist_ok=True)



def improve_robustness(img):
    
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_clahe = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    
    blur = cv2.GaussianBlur(img_clahe, (0, 0), sigmaX=IMG_SIZE / 30)
    img_final = cv2.addWeighted(img_clahe, 4, blur, -4, 128)
    return img_final

def get_wavelet_channels(img):
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    coeffs = pywt.dwt2(gray.astype(np.float32), 'haar')
    LL, (LH, HL, HH) = coeffs
    channels = []
    for subband in [LL, LH, HL, HH]:
        resized = cv2.resize(subband, (IMG_SIZE, IMG_SIZE))
       
        norm = (resized - resized.min()) / (resized.max() - resized.min() + 1e-6)
        channels.append(norm)
    return np.stack(channels, axis=-1)

def preprocess_image(path):
    
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
  
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 10
    if mask.any():
        coords = np.column_stack(np.where(mask))
        ymin, xmin = coords.min(axis=0); ymax, xmax = coords.max(axis=0)
        img = img[ymin:ymax+1, xmin:xmax+1]
    
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_enhanced = improve_robustness(img)
    
    
    wavelet_feats = get_wavelet_channels(img_enhanced)
    
    
    img_norm = img_enhanced.astype(np.float32) / 255.0
    return np.concatenate([img_norm, wavelet_feats], axis=-1)


def run_external_prep():
    X_ext, y_ext = [], []
    
    for folder_name, label_val in CLASSES_MAP.items():
        folder_path = os.path.join(RAW_DATASET2_DIR, folder_name)
        if not os.path.exists(folder_path):
            print(f"[Warning] directory cannot be found: {folder_path}")
            continue
            
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"Loading {folder_name} (label: {label_val})，total {len(files)} pictures...")
        
        for f in tqdm(files):
            img_p = os.path.join(folder_path, f)
            data = preprocess_image(img_p)
            if data is not None:
                X_ext.append(data)
                y_ext.append(label_val)

    X_ext = np.array(X_ext, dtype=np.float32)
    y_ext = np.array(y_ext)
    
    # 保存结果
    np.save(os.path.join(OUTPUT_DIR2, "X_dataset2.npy"), X_ext)
    np.save(os.path.join(OUTPUT_DIR2, "y_dataset2.npy"), y_ext)
    print(f"\n[Success] Data set 2 preprocessing completed，total {len(X_ext)}")
    print(f"file save to: {OUTPUT_DIR2}")

if __name__ == "__main__":
    run_external_prep()

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


DATA_DIR2 = "E:/2025-2026/Project/code1/Dataset2_processed"
MODEL_PATH = "./results_final/best_model.h5"
SAVE_DIR = "./results_external_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)

target_names = ["Normal", "Mild", "Moderate", "Severe", "Prolif"]


def focal_loss_with_smoothing(gamma=2., alpha=0.25, smoothing=0.1):
    def loss(y_true, y_pred):
        y_true = y_true * (1.0 - smoothing) + (smoothing / 5.0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        focal_weight = alpha * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(focal_weight * (-y_true * tf.math.log(y_pred)), axis=1)
    return loss

print("[INFO] Loading Model and Dataset 2...")
model = load_model(MODEL_PATH, custom_objects={'loss': focal_loss_with_smoothing()})
X_ext = np.load(os.path.join(DATA_DIR2, "X_dataset2.npy"))
y_ext_raw = np.load(os.path.join(DATA_DIR2, "y_dataset2.npy")) # 原始标签: 0=Normal, 3=DR


print("[INFO] Predicting and generating pseudo-labels...")
y_pred_probs = model.predict(X_ext, batch_size=16)
y_pred_classes = np.argmax(y_pred_probs, axis=1)


dr_indices = np.where(y_ext_raw > 0)[0]
normal_indices = np.where(y_ext_raw == 0)[0]


dr_predictions = y_pred_classes[dr_indices]
unique, counts = np.unique(dr_predictions, return_counts=True)
dist = dict(zip(unique, counts))


levels = [0, 1, 2, 3, 4]
counts_list = [dist.get(i, 0) for i in levels]

plt.figure(figsize=(10, 6))
colors = ['#66b3ff','#99ff99','#ffcc99','#ff9999','#ff6666']
bars = plt.bar(target_names, counts_list, color=colors, edgecolor='black', alpha=0.8)
plt.title('Distribution of Predicted Severity for Dataset 2 (DR Samples)', fontsize=14)
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)


for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 5, int(yval), ha='center', va='bottom')

plt.savefig(os.path.join(SAVE_DIR, "dr_severity_distribution.png"), dpi=300)
plt.show()


y_true_bin = (y_ext_raw > 0).astype(int)
y_pred_bin = (y_pred_classes > 0).astype(int)

bin_acc = accuracy_score(y_true_bin, y_pred_bin)
print(f"\n" + "="*40)
print(f"(Binary Accuracy): {bin_acc:.4f}")
print("="*40)


cm_bin = confusion_matrix(y_true_bin, y_pred_bin)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Predicted Normal", "Predicted DR"],
            yticklabels=["Actual Normal", "Actual DR"])
plt.title('Binary Classification Confusion Matrix (External Validation)')
plt.savefig(os.path.join(SAVE_DIR, "binary_confusion_matrix.png"), dpi=300)
plt.show()

print(f"\n[SUCCESS] Finised。{SAVE_DIR}")

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model


MODEL_PATH = "./results_final/best_model.h5"
DATA_DIR = "E:/2025-2026/Project/code1/APTOS_processed"
SAVE_DIR = "./gradcam_outputs"

os.makedirs(SAVE_DIR, exist_ok=True)


model = load_model(MODEL_PATH, compile=False)

X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("Loaded:", X_test.shape)


last_conv_layer_name = "conv2d" 


def gradcam(img, model, layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )

    img_tensor = tf.expand_dims(img, axis=0)

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor)
        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_out)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_out = conv_out[0]

    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


for i in range(len(X_test)):

    img = X_test[i]
    rgb = img[:, :, :3]

    heatmap = gradcam(img, model, last_conv_layer_name)

    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    rgb_uint8 = (rgb * 255).astype(np.uint8)

    overlay = cv2.addWeighted(rgb_uint8, 0.6, heatmap, 0.4, 0)

    save_path = os.path.join(SAVE_DIR, f"gradcam_{i}.png")
    cv2.imwrite(save_path, overlay)

    if i % 50 == 0:
        print(f"Processed {i}/{len(X_test)}")

print("Done. Saved to:", SAVE_DIR)

Basline CNN

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import pywt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


ROOT = "E:/2025-2026/Project/code1/archive/aptos_sorted"
IMG_SIZE = 224


def load_data_rgb():
    X, y = [], []

    for folder in os.listdir(ROOT):
        path = os.path.join(ROOT, folder)

        if folder == "NO_DR":
            label = 0
            files = os.listdir(path)

            for f in files:
                img = cv2.imread(os.path.join(path, f))
                if img is None: continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
                img = img.astype(np.float32)/255.0

                X.append(img)
                y.append(label)

        elif folder == "DR":
            mapping = {
                "Mild1":1,
                "Moderate2":2,
                "Severe4":3,
                "Proliferative3":4
            }

            for sub in mapping:
                sub_path = os.path.join(path, sub)
                for f in os.listdir(sub_path):
                    img = cv2.imread(os.path.join(sub_path,f))
                    if img is None: continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
                    img = img.astype(np.float32)/255.0

                    X.append(img)
                    y.append(mapping[sub])

    return np.array(X), tf.keras.utils.to_categorical(y,5)

# =========================
# Model
# =========================
def build_baseline():
    inputs = tf.keras.Input((224,224,3))

    x = tf.keras.layers.Conv2D(32,3,activation='relu',padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64,3,activation='relu',padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128,3,activation='relu',padding='same')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256,activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    outputs = tf.keras.layers.Dense(5,activation='softmax')(x)

    return tf.keras.Model(inputs,outputs)

# =========================
# Train
# =========================
X,y = load_data_rgb()
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = build_baseline()
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train,y_train,epochs=30,batch_size=16,validation_split=0.1)

pred = model.predict(X_test)
y_pred = np.argmax(pred,1)
y_true = np.argmax(y_test,1)

print("Baseline CNN")
print("Acc:",accuracy_score(y_true,y_pred))
print("F1:",f1_score(y_true,y_pred,average='macro'))
print("Kappa:",cohen_kappa_score(y_true,y_pred,weights='quadratic'))

Basline+CLAHE

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


ROOT = "E:/2025-2026/Project/code1/archive/aptos_sorted"
IMG_SIZE = 224


def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    img = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    return img


def load_data_clahe():
    X, y = [], []

    for folder in os.listdir(ROOT):
        path = os.path.join(ROOT, folder)

        # ================= NO_DR =================
        if folder == "NO_DR":
            label = 0

            for f in os.listdir(path):
                img = cv2.imread(os.path.join(path, f))
                if img is None:
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                
                img = apply_clahe(img)

                img = img.astype(np.float32) / 255.0

                X.append(img)
                y.append(label)

        # ================= DR =================
        elif folder == "DR":
            mapping = {
                "Mild1": 1,
                "Moderate2": 2,
                "Severe4": 3,
                "Proliferative3": 4
            }

            for sub in mapping:
                sub_path = os.path.join(path, sub)

                for f in os.listdir(sub_path):
                    img = cv2.imread(os.path.join(sub_path, f))
                    if img is None:
                        continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                    # ⭐ CLAHE only
                    img = apply_clahe(img)

                    img = img.astype(np.float32) / 255.0

                    X.append(img)
                    y.append(mapping[sub])

    return np.array(X), tf.keras.utils.to_categorical(y, 5)


def build_model():
    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    outputs = tf.keras.layers.Dense(5, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)


X, y = load_data_clahe()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=np.argmax(y, axis=1)
)

model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.1
)


pred = model.predict(X_test)

y_pred = np.argmax(pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\n===== CLAHE CNN Results =====")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average='macro'))
print("Kappa:", cohen_kappa_score(y_true, y_pred, weights='quadratic'))

Basline+CLAHE+Wavelet

In [ ]:
import os
import cv2
import numpy as np
import pywt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


ROOT = "E:/2025-2026/Project/code1/archive/aptos_sorted"
IMG_SIZE = 224


def get_wavelet(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    coeffs = pywt.dwt2(gray.astype(np.float32), 'haar')
    LL, (LH, HL, HH) = coeffs

    channels = []
    for band in [LL, LH, HL, HH]:
        band = cv2.resize(band, (IMG_SIZE, IMG_SIZE))
        band = (band - band.min()) / (band.max() - band.min() + 1e-6)
        channels.append(band)

    return np.stack(channels, axis=-1)


def load_wavelet_dataset():
    X, y = [], []

    for folder in os.listdir(ROOT):
        path = os.path.join(ROOT, folder)

        # ================= NO_DR =================
        if folder == "NO_DR":
            label = 0

            for f in os.listdir(path):
                img = cv2.imread(os.path.join(path, f))
                if img is None:
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                wave = get_wavelet(img)

                rgb = img.astype(np.float32) / 255.0

                X.append(np.concatenate([rgb, wave], axis=-1))
                y.append(label)

        # ================= DR =================
        elif folder == "DR":
            mapping = {
                "Mild1": 1,
                "Moderate2": 2,
                "Severe4": 3,
                "Proliferative3": 4
            }

            for sub in mapping:
                sub_path = os.path.join(path, sub)

                for f in os.listdir(sub_path):
                    img = cv2.imread(os.path.join(sub_path, f))
                    if img is None:
                        continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                    wave = get_wavelet(img)
                    rgb = img.astype(np.float32) / 255.0

                    X.append(np.concatenate([rgb, wave], axis=-1))
                    y.append(mapping[sub])

    return np.array(X), tf.keras.utils.to_categorical(y, 5)


def build_wavelet_model():
    inputs = tf.keras.Input(shape=(224, 224, 7))

    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    outputs = tf.keras.layers.Dense(5, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)


X, y = load_wavelet_dataset()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=np.argmax(y, axis=1)
)


model = build_wavelet_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.1
)


pred = model.predict(X_test)

y_pred = np.argmax(pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\n===== Wavelet CNN Results =====")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average='macro'))
print("Kappa:", cohen_kappa_score(y_true, y_pred, weights='quadratic'))